In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 0.12 Interpolation: Polynomials, Runge's Warning, and Splines

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume 0 — Mathematical & Computational Foundations",
    number="0.12",
    title="Interpolation: Polynomials, Runge's Warning, and Splines",
    blurb="Connecting the dots, done right: the one polynomial through "
    "n+1 points, the famous bell curve that punishes equispaced nodes "
    "with divergence, the Chebyshev fix and the Lebesgue constant that "
    "explains it, cubic splines whose fourth-order accuracy hangs on the "
    "boundary condition, and the Simpson weights of §0.3 rederived from "
    "where they always came from.",
    difficulty="introductory",
    estimate="100–130 min",
)

## Notebook overview

Between every table of data and every formula evaluated on a grid
stands the same question: what happens *between* the points?
Interpolation is the discipline of answering it honestly, and it is
quietly load-bearing for half of this course — the Newton–Cotes
quadrature weights of [§0.3](quadrature-differentiation.ipynb) are
integrals of interpolating polynomials, finite differences are their
derivatives, and every plotted curve through computed samples is an
interpolant someone chose not to think about.

The subject's central drama is a genuine surprise. Polynomial
interpolation through $n+1$ points is unique and, on paper, converges
beautifully — yet for a function as innocent as Runge's
$1/(1 + 25x^2)$, interpolating at *equally spaced* nodes diverges,
violently, as $n$ grows. The failure is not roundoff and not the
polynomial's fault; it is the *nodes'* fault, quantified by a single
number (the Lebesgue constant) that we will measure. The two cures —
Chebyshev nodes, which tame the constant to a logarithm, and piecewise
splines, which abandon high degree altogether — between them cover
nearly every interpolation a physicist ever does. Numerical Recipes
{cite}`numrecipes` and Trefethen {cite}`trefethen1997` are the
standing references.

A note on reading the checks in this notebook: a validation compares a
result to an expected physical fact. A ✗ does not by itself mean the
answer is wrong; it means the output did not match what the check
expected, which may be a genuine error, a different-but-valid
convention, or too tight a tolerance. Treat a ✗ as a prompt to locate
the discrepancy. Passing is strong evidence, not proof.

## Theory in brief

**The interpolating polynomial.** Through any $n+1$ points
$(x_i, y_i)$ with distinct nodes there passes exactly one polynomial
of degree $\le n$. Lagrange's form makes existence explicit,

```{math}
:label: eq-ip-lagrange
p_n(x) \;=\; \sum_{i=0}^{n} y_i\,\ell_i(x),
\qquad
\ell_i(x) \;=\; \prod_{j\ne i}\frac{x - x_j}{x_i - x_j},
```

with the cardinal property $\ell_i(x_j) = \delta_{ij}$. Evaluating
{eq}`eq-ip-lagrange` well is its own craft: the *barycentric* form
(algebraically identical, numerically stable) is what
`scipy.interpolate.BarycentricInterpolator` implements, and it
succeeds at degrees where the "obvious" route — solving the
Vandermonde system for coefficients — has long since drowned in
conditioning.

**The error, and the nodes' role in it.** For $f$ with $n+1$
derivatives,

```{math}
:label: eq-ip-error
f(x) - p_n(x) \;=\;
\frac{f^{(n+1)}(\xi)}{(n+1)!}\,\prod_{i=0}^{n}(x - x_i) ,
```

and everything the interpolator can control lives in the node
polynomial $\prod(x - x_i)$. Equispaced nodes make it explode near the
interval's ends; Chebyshev nodes $x_i = \cos\bigl((2i+1)\pi/(2n+2)\bigr)$
equidistribute it. The clean diagnostic is the **Lebesgue constant**
$\Lambda_n = \max_x \sum_i |\ell_i(x)|$: the interpolant's error is at
most $(1 + \Lambda_n)$ times the *best possible* polynomial error, so
$\Lambda_n$ is the price of interpolating instead of optimizing. For
equispaced nodes $\Lambda_n$ grows like $2^n/(n\log n)$ — catastrophe
— while for Chebyshev nodes

```{math}
:label: eq-ip-lebesgue
\Lambda_n^{\rm Cheb} \;\approx\; \frac{2}{\pi}\ln(n+1) + 0.9625 ,
```

barely worse than optimal. Runge's phenomenon is this dichotomy made
visible on one bell-shaped function.

**Splines: low degree, many pieces.** The alternative to fighting
high-degree polynomials is refusing to use them: a **cubic spline**
interpolates with a separate cubic on each subinterval, glued to
$C^2$ smoothness. For a smooth $f$ on a grid of spacing $h$ the error
is $O(h^4)$ — *if* the two leftover degrees of freedom (the boundary
conditions) are spent wisely. The default "not-a-knot" condition
preserves fourth order; the innocently named "natural" spline
($S'' = 0$ at the ends) forces a curvature the true function rarely
has, and degrades the *global* order to $O(h^2)$. The choice of
boundary condition is worth two orders of accuracy, which is the kind
of fact one prefers to learn on a test function rather than on data.

## Setup

Everything runs on $[-1, 1]$ (or $[0, 1]$ for the spline tests) with
pinned node counts and a fixed $5001$-point evaluation grid for
max-norm errors. Nothing is stochastic.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import BarycentricInterpolator, CubicSpline

from ecp import validate

X_EVAL = np.linspace(-1.0, 1.0, 5001)


def runge(x):
    """Runge's function 1/(1 + 25 x^2).

    The innocent-looking bell curve on [-1, 1] whose equispaced
    interpolants diverge as the degree grows: the standard cautionary
    example of the subject.

    Parameters
    ----------
    x : float or numpy.ndarray
        Evaluation point(s).

    Returns
    -------
    float or numpy.ndarray
        Function value(s).
    """
    return 1.0 / (1.0 + 25.0 * x**2)


def chebyshev_nodes(n):
    """The n+1 Chebyshev nodes cos((2i+1)π/(2n+2)) on [-1, 1].

    The roots of the Chebyshev polynomial T_{n+1}: nodes clustered
    toward the endpoints exactly strongly enough to equidistribute the
    node polynomial of eq-ip-error.

    Parameters
    ----------
    n : int
        Polynomial degree (returns n+1 nodes).

    Returns
    -------
    numpy.ndarray
        The nodes, in decreasing order.
    """
    i = np.arange(n + 1)
    return np.cos((2.0 * i + 1.0) * np.pi / (2.0 * n + 2.0))


def max_interp_error(f, nodes):
    """Max-norm interpolation error of f at the given nodes.

    Builds the barycentric interpolant through (nodes, f(nodes)) and
    measures max |p - f| on the fixed 5001-point grid.

    Parameters
    ----------
    f : callable
        Function to interpolate.
    nodes : numpy.ndarray
        Interpolation nodes in [-1, 1].

    Returns
    -------
    float
        The maximum absolute error.
    """
    p = BarycentricInterpolator(nodes, f(nodes))
    return float(np.max(np.abs(p(X_EVAL) - f(X_EVAL))))


def lebesgue_constant(nodes):
    """The Lebesgue constant max_x sum_i |l_i(x)| of a node set.

    Assembles each Lagrange cardinal polynomial l_i explicitly by its
    product formula and maximizes the sum of absolute values on the
    evaluation grid: the amplification factor between the best possible
    polynomial error and the interpolant's.

    Parameters
    ----------
    nodes : numpy.ndarray
        Interpolation nodes.

    Returns
    -------
    float
        The Lebesgue constant.
    """
    total = np.zeros_like(X_EVAL)
    for i in range(len(nodes)):
        li = np.ones_like(X_EVAL)
        for j in range(len(nodes)):
            if j != i:
                li *= (X_EVAL - nodes[j]) / (nodes[i] - nodes[j])
        total += np.abs(li)
    return float(total.max())

## Exercise 1 — One polynomial, certified

{eq}`eq-ip-lagrange` makes two exact promises: the interpolant passes
through its data, and it *is* any polynomial of degree $\le n$ that
generated the data.

**Part a)** Interpolate the polynomial $q(x) = 3x^5 - 2x^3 + x - 1$
at the $6$ equispaced nodes `numpy.linspace(-1, 1, 6)` with
`scipy.interpolate.BarycentricInterpolator` and verify the interpolant
reproduces $q$ *everywhere* — max-norm difference on the evaluation
grid below $10^{-12}$ — because a degree-$5$ interpolant of degree-$5$
data has nowhere else to go. Verify also the cardinal property: the
interpolant of the data $y = (0,0,1,0,0,0)$ equals $1$ at the third
node and $0$ at the other five (`atol=1e-13`).

**Part b)** Verify uniqueness numerically: interpolate the same $q$
at $6$ *Chebyshev* nodes and verify the two interpolants — different
nodes, same data-generating polynomial — agree with each other to
$10^{-12}$ on the whole grid. Same polynomial, because there is only
one.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    err_exact < 1e-12,
    "a degree-5 interpolant reproduces degree-5 data everywhere: it "
    "has nowhere else to go",
    f"max error {err_exact:.1e}",
)
validate.check(
    abs(card_at_nodes[2] - 1.0) < 1e-13
    and max(abs(card_at_nodes[j]) for j in (0, 1, 3, 4, 5)) < 1e-13,
    "the cardinal property holds: one at its own node, zero at the " "others",
    f"values {np.round(card_at_nodes, 14)}",
)
validate.check(
    uniq_gap < 1e-12,
    "and two different node sets give the SAME interpolant of the same "
    "polynomial: uniqueness, observed",
    f"gap {uniq_gap:.1e}",
)

## Exercise 2 — Runge's warning, measured

Now the drama. Runge's function $1/(1 + 25x^2)$ is smooth, bounded,
and bell-shaped — and equispaced interpolation of it *diverges*.

**Part a)** Measure the max-norm error of interpolating `runge` at
equispaced nodes for degrees $n = 10$ and $n = 20$: verify the errors
are $1.916$ and $59.8$ (`rtol=1e-2` each) — the error *grew* thirty-
fold when the data doubled, on a function with no singularity on the
real line. (The culprit, via {eq}`eq-ip-error`, is the node
polynomial's end-of-interval explosion, amplified by the poles of
$f$ at $\pm i/5$ in the complex plane.)

**Part b)** The cure: the same degrees at Chebyshev nodes give errors
$0.109$ and $0.0153$ (`rtol=1e-2` each) — monotone convergence where
equispaced diverged. Plot both interpolants at $n = 20$ over the true
function: the equispaced one swinging to $\pm 60$ at the edges, the
Chebyshev one indistinguishable from $f$ at plot scale.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([errs_eq[10], errs_eq[20]]),
    np.array([ERR_TARGETS_EQ[10], ERR_TARGETS_EQ[20]]),
    "equispaced interpolation of Runge's function DIVERGES: doubling "
    "the data multiplied the error thirty-fold",
    rtol=1e-2,
)
validate.close(
    np.array([errs_ch[10], errs_ch[20]]),
    np.array([ERR_TARGETS_CH[10], ERR_TARGETS_CH[20]]),
    "while Chebyshev nodes converge on the same function at the same "
    "degrees: the nodes, not the polynomial, were the problem",
    rtol=1e-2,
)

## Exercise 3 — The Lebesgue constant: the price of a node set

{eq}`eq-ip-lebesgue` promised a single number that explains Exercise
2, and it is checkable to four digits.

**Part a)** Compute $\Lambda_n$ with `lebesgue_constant` for
equispaced and Chebyshev nodes at $n = 10$ and $20$. Verify the
Chebyshev values match the asymptotic formula
$\tfrac{2}{\pi}\ln(n+1) + 0.9625$ to `rtol=1e-3` at both degrees
($2.489$ and $2.901$: barely growing), and verify the equispaced
catastrophe: $\Lambda_{10} = 29.9$ (`rtol=1e-2`) and
$\Lambda_{20} > 10^4$ — a four-order-of-magnitude amplifier by degree
twenty.

**Part b)** Close the logic: the interpolation error is bounded by
$(1 + \Lambda_n)$ times the best-approximation error, so Chebyshev
interpolation is within a factor $\approx 4$ of the best possible
degree-$20$ polynomial while equispaced interpolation may be $10^4$
times worse. Verify the bound holds in practice for Runge at
$n = 20$: the measured Chebyshev error $0.0153$ is *less* than
$(1 + \Lambda_{20}^{\rm Cheb})$ times the best-approximation error,
estimated from below by half the Chebyshev error itself — an
admittedly circular floor, so verify the sharper, honest statement:
the ratio of equispaced to Chebyshev errors ($59.8/0.0153 \approx
3900$) is within a factor of $3$ of the ratio of their Lebesgue
constants ($10986/2.90 \approx 3790$): the constants *predict* the
disaster's magnitude, not just its sign.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([leb_ch[10], leb_ch[20]]),
    np.array([leb_formula[10], leb_formula[20]]),
    "the Chebyshev Lebesgue constants sit on the (2/π)ln(n+1) + 0.9625 "
    "asymptote to four digits: logarithmic, tame, nearly optimal",
    rtol=1e-3,
)
validate.check(
    abs(leb_eq[10] - 29.9) / 29.9 < 1e-2 and leb_eq[20] > 1e4,
    "while the equispaced constants explode past ten thousand by " "degree twenty",
    f"Λ10 = {leb_eq[10]:.1f}, Λ20 = {leb_eq[20]:.3g}",
)
validate.check(
    1.0 / 3.0 < (ratio_err / ratio_leb) < 3.0,
    "and the constants PREDICT the disaster's size: the error ratio "
    "tracks the Lebesgue ratio within a factor of three",
    f"errors {ratio_err:.0f}× vs constants {ratio_leb:.0f}×",
)

## Exercise 4 — Conditioning: why the barycentric form earns its keep

The textbook route to the interpolating polynomial — solve the
Vandermonde system $V\mathbf c = \mathbf y$ for monomial coefficients
— is a conditioning time bomb, and degree $40$ is past the detonation.

**Part a)** Build the Vandermonde matrix `numpy.vander` for the $41$
equispaced nodes on $[-1, 1]$ and verify its condition number
(`numpy.linalg.cond`) exceeds $10^{17}$: beyond double precision,
meaning the monomial coefficients are pure noise regardless of
solver.

**Part b)** Verify the barycentric evaluation sails on regardless:
interpolating `runge` at the $41$ *Chebyshev* nodes via
`BarycentricInterpolator` gives a max-norm error below $10^{-3}$
(continuing Exercise 2's convergence right through the degree where
the monomial route died). The lesson generalizes far beyond
interpolation: the same mathematics in a different basis is a
different algorithm.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    cond_v > 1e17,
    "the degree-40 Vandermonde matrix is conditioned beyond double "
    "precision: monomial coefficients are noise",
    f"cond = {cond_v:.2e}",
)
validate.check(
    err_ch40 < 1e-3,
    "while the barycentric form at the same degree keeps converging: "
    "same polynomial, different basis, different algorithm",
    f"error {err_ch40:.1e}",
)

## Exercise 5 — Cubic splines and the two-orders boundary condition

The spline route abandons high degree for many low-degree pieces, and
its one subtlety is worth two orders of accuracy.

**Part a)** Interpolate $e^x$ on $[0, 1]$ with
`scipy.interpolate.CubicSpline` at $n = 8, 16, 32, 64, 128$
subintervals under the default not-a-knot boundary condition, measure
max-norm errors on a $4001$-point grid, and verify fourth-order
convergence: the `numpy.polyfit` slope of $\log(\rm error)$ against
$\log h$ in $[3.8, 4.2]$, with the finest error below $10^{-9}$.

**Part b)** Repeat with `bc_type="natural"` — the spline forced to
$S'' = 0$ at both ends, where $e^x$ has $e^0 = 1$ and $e^1 = 2.72$ —
and verify the degradation: slope in $[1.9, 2.1]$, and the finest
error more than $10^4$ times worse than not-a-knot's. "Natural" names
a variational property (minimum curvature), not a recommendation; the
false boundary curvature pollutes the whole interval at $O(h^2)$.

**Part c)** The birthplace payoff. The Simpson weights that
[§0.3](quadrature-differentiation.ipynb) tabulated are integrals of
{eq}`eq-ip-lagrange`'s cardinal polynomials: for the three nodes
$0, 1, 2$ (spacing $h = 1$), fit each cardinal's coefficients with
`numpy.polyfit` (degree 2), integrate with `numpy.polyint`, and verify
the three integrals equal $(\tfrac13, \tfrac43, \tfrac13)$ to
`atol=1e-12`: quadrature is interpolation, integrated — the circle
from this notebook back to [§0.3](quadrature-differentiation.ipynb),
closed.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    3.8 < slope_nak < 4.2 and errs_nak[-1] < 1e-9,
    "the not-a-knot cubic spline converges at fourth order, reaching "
    "1e-10 territory by h = 1/128",
    f"slope {slope_nak:.2f}, finest {errs_nak[-1]:.1e}",
)
validate.check(
    1.9 < slope_nat < 2.1 and errs_nat[-1] > 1e4 * errs_nak[-1],
    "the 'natural' condition costs two orders globally: a boundary "
    "condition is an accuracy decision",
    f"slope {slope_nat:.2f}, penalty ×{errs_nat[-1] / errs_nak[-1]:.0f}",
)
validate.close(
    simpson_w,
    np.array([1.0 / 3.0, 4.0 / 3.0, 1.0 / 3.0]),
    "and Simpson's (1/3, 4/3, 1/3) fall out of integrating the "
    "Lagrange cardinals: quadrature is interpolation, integrated",
    rtol=0.0,
    atol=1e-12,
)

## Notebook summary

- The interpolating polynomial behaved as its theorems promise:
  degree-5 data reproduced to $10^{-13}$, cardinal functions exactly
  cardinal, and two node sets yielding the *same* (unique)
  interpolant.
- Runge's phenomenon arrived on schedule: equispaced errors $1.9 \to
  59.8$ as $n$ went $10 \to 20$ on a smooth bell curve, while
  Chebyshev nodes converged $0.109 \to 0.0153$ on the same function
  with the same algorithm.
- The Lebesgue constant priced it: Chebyshev's $2.489$ and $2.901$ on
  the logarithmic asymptote to four digits, equispaced past $10^4$ by
  degree twenty — and the error ratio *tracked* the constant ratio
  within a factor of three.
- Conditioning separated mathematics from algorithm: the degree-40
  Vandermonde matrix at $\mathrm{cond} > 10^{17}$ while barycentric
  evaluation converged serenely below $10^{-3}$.
- Splines delivered fourth order under not-a-knot and lost exactly
  two orders under "natural" boundary conditions ($\times 10^4$ in
  error at $h = 1/128$); and Simpson's weights fell out of
  integrating the cardinals — the quadrature of
  [§0.3](quadrature-differentiation.ipynb), returned to its
  birthplace.

## Outlook

- **Spectral methods.** Push Chebyshev interpolation to its limit and
  differentiation of the interpolant becomes a dense matrix applied
  to samples: spectral collocation, with exponential accuracy for
  analytic functions {cite}`trefethen1997` — the method behind many
  of the smoothest solvers in computational physics.
- **Fitting is not interpolating.** With noisy data, passing through
  every point means interpolating the noise;
  [§0.8](fitting-least-squares.ipynb) is the other regime, and
  knowing which side of the divide a dataset sits on is a scientist's
  judgment call.
- **Higher dimensions.** Tensor-product grids inherit everything
  here; scattered data needs new ideas (radial basis functions,
  triangulation) — the machinery inside every contour plot this
  course draws.
- **Splines beyond graphs.** The same $C^2$ piecewise cubics
  parametrize fonts, roads, and robot trajectories; the "natural"
  spline's minimum-curvature property, useless for accuracy, is
  exactly what a draftsman's bending strip wants.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()